# 03 - Model Training

Trains Logistic Regression, Random Forest, and GBT classifiers and reports
AUC and AUPR on a held-out test split.

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from src.utils import get_spark, load_config
from src import data_preprocessing as dp
from src.models import build_model

cfg = load_config('../config/config.yaml')
spark = get_spark()
featurized = spark.read.parquet(cfg['data']['output_path'] + 'featurized/')
train, test = dp.split_train_test(featurized, cfg['model']['test_size'], cfg['model']['random_state'])

In [ ]:
auc = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC')
aupr = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderPR')

for name in ('logistic_regression', 'random_forest', 'gbt'):
    estimator = build_model(name)
    model = estimator.fit(train)
    preds = model.transform(test)
    print(f'{name}: AUC={auc.evaluate(preds):.4f} | AUPR={aupr.evaluate(preds):.4f}')
    model.write().overwrite().save(cfg['data']['output_path'] + f'models/{name}/')
    preds.write.mode('overwrite').parquet(cfg['data']['output_path'] + f'predictions/{name}/')